# 02. Treinar Splink (link_only Censo × CPF)

Treina no conjunto inteiro e grava `splink_model.json`. Spec (comparisons,
prior, EM) nas células abaixo. Predict: [`02b_aplicar_splink.ipynb`](02b_aplicar_splink.ipynb).
Avaliação: [`03_avaliar.ipynb`](03_avaliar.ipynb).

`cpf_norm` só no prior/EM. Score de nome: `nome_completo_phon`,
`primeiro_nome_phon` e `ultimo_nome_phon` (meio e o composto `primeiro_ultimo`
ficam nas tabelas; meio não entra no comparison). Blocking de predição (12
regras) está no [`02b_aplicar_splink.ipynb`](02b_aplicar_splink.ipynb). Aqui
só prior e EM.


In [ ]:
import sys
from pathlib import Path

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

from config import (
    DUCKDB_MEMORY_LIMIT,
    DUCKDB_THREADS,
    SPLINK_INPUT_VIEW,
    SPLINK_MODEL_JSON,
    TABELA_CENSO_LIMPA,
    TABELA_CPF_LIMPA,
    drop_splink_temp_tables,
    get_connection,
    get_splink_db_api,
    materialize_splink_input,
    print_paths,
    require_tables,
)

print_paths()
con = get_connection()
drop_splink_temp_tables(con)
require_tables(con, [TABELA_CENSO_LIMPA, TABELA_CPF_LIMPA], notebook_origem='00b')
materialize_splink_input(con)
db_api = get_splink_db_api(con)

n_reg = con.execute(f'SELECT COUNT(*) FROM {SPLINK_INPUT_VIEW}').fetchone()[0]
duck_settings = con.execute(
    "SELECT current_setting('threads'), current_setting('memory_limit')"
).fetchone()
print(f'Registros: {n_reg:,}')
print(
    f'DuckDB: threads={duck_settings[0]}, memory_limit={duck_settings[1]} '
    f'(defaults: {DUCKDB_THREADS}, {DUCKDB_MEMORY_LIMIT})'
)

SPLINK_ANALYSIS_SAMPLE_N = 1_000_000
U_MAX_PAIRS = 100_000_000
analysis_table = SPLINK_INPUT_VIEW
if n_reg > SPLINK_ANALYSIS_SAMPLE_N:
    con.execute(f'''
    CREATE OR REPLACE TEMP TABLE splink_analysis_sample AS
    SELECT * FROM {SPLINK_INPUT_VIEW}
    USING SAMPLE {SPLINK_ANALYSIS_SAMPLE_N} ROWS
    ''')
    analysis_table = 'splink_analysis_sample'
    print(
        f'Amostra só para profile (não entra no Linker): '
        f'{SPLINK_ANALYSIS_SAMPLE_N:,} de {n_reg:,}'
    )


## Sanity check - composição da base

Volume por fonte e preenchimento das colunas usadas no linkage. Blocking em
coluna muito vazia gera poucos pares candidatos.


In [ ]:
from IPython.display import display

display(con.execute(f'''
SELECT origem, COUNT(*) AS n
FROM {SPLINK_INPUT_VIEW} GROUP BY 1 ORDER BY 2 DESC
''').df())

LINKAGE_COLS = [
    'primeiro_nome', 'ultimo_nome', 'nome_completo','nome_meio',
    'primeiro_ultimo',
    'nome_mae', 'primeiro_nome_mae', 'nome_meio_mae', 'ultimo_nome_mae',
    'data_nascimento', 'ano_nascimento', 'mes_nascimento', 'dia_nascimento',
    'idade', 'cep', 'sexo', 'uf',
    'nome_completo_phon','primeiro_nome_phon','nome_meio_phon','ultimo_nome_phon',
    'primeiro_ultimo_phon',
    'nome_mae_phon','primeiro_nome_mae_phon','nome_meio_mae_phon','ultimo_nome_mae_phon'
]

cols_presentes = [
    c for c in LINKAGE_COLS
    if c in set(con.execute(f'SELECT * FROM {SPLINK_INPUT_VIEW} LIMIT 0').df().columns)
]
preenchimento = ',\n    '.join(
    f"ROUND(100.0 * COUNT({c}) / COUNT(*), 1) AS pct_{c}" for c in cols_presentes
)
display(con.execute(f'''
SELECT origem, {preenchimento}
FROM {SPLINK_INPUT_VIEW} GROUP BY origem ORDER BY origem
''').df().T)


## Exploração pré-modelo

Profile Splink das colunas de linkage (amostra se a base passar de
`SPLINK_ANALYSIS_SAMPLE_N` — o `profile_columns` materializa pandas).
O treino usa o conjunto inteiro. Volume de candidatos do `predict` é o gráfico
de blocking no [`02b_aplicar_splink.ipynb`](02b_aplicar_splink.ipynb).


In [ ]:
from splink.exploratory import profile_columns

profile_columns(
    con.execute(f'SELECT * FROM {analysis_table}').df(),
    db_api,
    column_expressions=[
        'primeiro_nome_phon', 'nome_meio_phon', 'ultimo_nome_phon', 'nome_completo_phon',
        'primeiro_ultimo_phon',
        'uf', 'data_nascimento', 'ano_nascimento', 'mes_nascimento', 'dia_nascimento',
        'idade', 'cpf_norm',
    ],
)


## Modelo Splink

`Linker` nas views `splink_censo` / `splink_cpf` = **todas** as linhas de
`censo_limpo` / `cpf_limpo` (assert na célula do Linker). Settings em `link_only`.

Comparisons neste notebook: nomes fonéticos (`nome_completo_phon`,
`primeiro_nome_phon` e `ultimo_nome_phon`; JW 0,95 e 0,92 + TF), data (Null
custom → Exact+TF → Damerau ≤ 1 → mês e dia iguais → ELSE `m=1e-6` fixo),
idade (exact e ±1; ELSE `m=1e-6` fixo, como a data — só pesa quando a idade
não é nula), UF.
Sem `nome_mae*`, sem sexo, sem CEP e **sem CPF** no score. Meio e o composto
`primeiro_ultimo*` ficam nas tabelas; não entram no comparison.
CEP entra no EM; sexo só no EM `sexo+DOB+UF+CEP`. As 12 regras de `predict`
(CEP/sexo na predição) ficam no 02b.
`cpf_norm` entra no prior e num EM; `NULL = NULL` é falso, então Censo sem ouro
não fabrica par.


In [ ]:
import splink.comparison_level_library as cll
import splink.comparison_library as cl

JW_NOME = [0.95, 0.92]

null_sql = (
    '(data_nascimento_l IS NULL OR data_nascimento_r IS NULL) AND '
    '(mes_nascimento_l IS NULL OR mes_nascimento_r IS NULL OR '
    'dia_nascimento_l IS NULL OR dia_nascimento_r IS NULL)'
)
mes_dia_sql = (
    'mes_nascimento_l = mes_nascimento_r AND '
    'dia_nascimento_l = dia_nascimento_r AND '
    'mes_nascimento_l IS NOT NULL AND dia_nascimento_l IS NOT NULL'
)

comparisons = [
    cl.NameComparison(
        'nome_completo_phon', jaro_winkler_thresholds=JW_NOME
    ).configure(term_frequency_adjustments=True),
    cl.NameComparison(
        'primeiro_nome_phon', jaro_winkler_thresholds=JW_NOME
    ).configure(term_frequency_adjustments=True),
    cl.NameComparison(
        'ultimo_nome_phon', jaro_winkler_thresholds=JW_NOME
    ).configure(term_frequency_adjustments=True),
    cl.CustomComparison(
        comparison_levels=[
            cll.CustomLevel(null_sql, label_for_charts='Null data e mes/dia').configure(
                is_null_level=True
            ),
            cll.ExactMatchLevel('data_nascimento', term_frequency_adjustments=True),
            cll.DamerauLevenshteinLevel('data_nascimento', 1),
            cll.CustomLevel(mes_dia_sql, label_for_charts='Mes e dia iguais'),
            cll.ElseLevel().configure(
                m_probability=1e-6,
                fix_m_probability=True,
            ),
        ],
        output_column_name='data_nascimento',
    ),
    cl.CustomComparison(
        comparison_levels=[
            cll.NullLevel('idade'),
            cll.ExactMatchLevel('idade'),
            cll.AbsoluteDifferenceLevel('idade', 1),
            cll.ElseLevel().configure(
                m_probability=1e-6,
                fix_m_probability=True,
            ),
        ],
        output_column_name='idade',
    ),
    cl.ExactMatch('uf').configure(term_frequency_adjustments=True),
]


In [ ]:
from splink import Linker, SettingsCreator

con.execute(f'''
CREATE OR REPLACE VIEW splink_censo AS
SELECT * FROM {SPLINK_INPUT_VIEW} WHERE origem = 'censo'
''')
con.execute(f'''
CREATE OR REPLACE VIEW splink_cpf AS
SELECT * FROM {SPLINK_INPUT_VIEW} WHERE origem = 'cpf'
''')
n_censo_limpo = con.execute(f'SELECT COUNT(*) FROM {TABELA_CENSO_LIMPA}').fetchone()[0]
n_cpf_limpo = con.execute(f'SELECT COUNT(*) FROM {TABELA_CPF_LIMPA}').fetchone()[0]
n_censo_view = con.execute('SELECT COUNT(*) FROM splink_censo').fetchone()[0]
n_cpf_view = con.execute('SELECT COUNT(*) FROM splink_cpf').fetchone()[0]
assert n_censo_view == n_censo_limpo and n_cpf_view == n_cpf_limpo, (
    f'Linker não está no conjunto inteiro: '
    f'censo {n_censo_view} vs limpo {n_censo_limpo}; '
    f'cpf {n_cpf_view} vs limpo {n_cpf_limpo}'
)
print('Linker: conjunto inteiro', f'{n_censo_view:,}', f'{n_cpf_view:,}')

settings = SettingsCreator(
    link_type='link_only',
    unique_id_column_name='unique_id',
    comparisons=comparisons,
    retain_intermediate_calculation_columns=False,
)
linker = Linker(
    ['splink_censo', 'splink_cpf'],
    settings,
    db_api=db_api,
    input_table_aliases=['censo', 'cpf'],
)


## Prior

Outra lista, distinta da predição (02b): `nome_completo+DOB` e `cpf_norm`. Estima a
probabilidade a priori de dois registros aleatórios casarem. O `u` vem de
pares **aleatórios** (`U_MAX_PAIRS`).


In [ ]:
from splink import block_on

deterministic_rules = [
    block_on('cpf_norm'),
    block_on('nome_completo_phon', 'data_nascimento'),    
]

linker.training.estimate_probability_two_random_records_match(deterministic_rules, recall=0.7)
# Amostra de pares aleatórios para o parâmetro u (não são os pares do blocking).
print('max_pairs u:', f'{U_MAX_PAIRS:,}')
linker.training.estimate_u_using_random_sampling(max_pairs=U_MAX_PAIRS)


## EM

Quatro blocos **apertados e separados**, cada um com mistura match/não-match
para estimar `m`. Não são as 12 regras de predição (essas estão no 02b). O bloco
`primeiro+DOB` observa discordância de `ultimo_nome_phon` e do completo.
`cpf_norm` só aqui e no prior.


In [ ]:
# EM em CPF ouro 1:1: bloco quase só match (u permanece o do random sampling).
# cpf_norm só no EM/prior; as 12 regras de predict ficam no 02b.
linker.training.estimate_parameters_using_expectation_maximisation(
    block_on('cpf_norm'),
    estimate_without_term_frequencies=True,
)


In [ ]:
# EM em sexo+DOB (sem CEP): observa discordância de nome.
linker.training.estimate_parameters_using_expectation_maximisation(
    block_on('sexo','data_nascimento','uf',"cep"),
    estimate_without_term_frequencies=True,
)


In [ ]:
# EM em primeiro_nome_phon + DOB: observa m de ultimo_nome_phon, completo, idade.
linker.training.estimate_parameters_using_expectation_maximisation(
    block_on('primeiro_nome_phon', 'data_nascimento'),
    estimate_without_term_frequencies=True,
)


In [ ]:
# EM em nome + sexo + UF + CEP: observa m de data/idade.
linker.training.estimate_parameters_using_expectation_maximisation(
    block_on('primeiro_nome_phon', 'ultimo_nome_phon','sexo','uf','cep'),
    estimate_without_term_frequencies=True,
)

In [ ]:
from config import MODELS_DIR

SPLINK_MODEL_JSON.parent.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
linker.misc.save_model_to_json(str(SPLINK_MODEL_JSON), overwrite=True)
linker.misc.save_model_to_json(str(MODELS_DIR / 'splink_model.json'), overwrite=True)
print('Modelo salvo:', SPLINK_MODEL_JSON)
print('Cópia em models/:', MODELS_DIR / 'splink_model.json')


## Visualização pós-treino

Match weights e registros difíceis de linkar (unlinkables).


In [ ]:
linker.visualisations.match_weights_chart()


In [ ]:
linker.evaluation.unlinkables_chart()


## Encerrar

JSON pronto para o [`02b_aplicar_splink.ipynb`](02b_aplicar_splink.ipynb).


In [ ]:
print(f"{'modelo':12s} {'ok ' if SPLINK_MODEL_JSON.exists() else 'FALTA'} {SPLINK_MODEL_JSON}")


In [ ]:
con.close()
